# Phase F1 — Auto-label ~37k Warhammer images on Colab

**What you do (once):**
1. Upload `photoanalyzer_f1_bundle.tar` to the *root* of your Google Drive (MyDrive).
2. Runtime → Change runtime type → **T4 GPU**.
3. Runtime → Run all.
4. Walk away. The notebook is resumable — if Colab disconnects, re-run and it picks up where it left off.
5. When finished, download `f1_outputs.tar` from Drive to your repo, then `tar -xf f1_outputs.tar` (yields `data/pseudo_labels/`).

Expected runtime on T4: **~10 hours** for 37k images. Drive output is checkpointed every 200 images.

## 1. Mount Drive + verify GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), (
    'No GPU attached. Runtime → Change runtime type → T4 GPU, then Runtime → Run all.'
)
print('GPU:', torch.cuda.get_device_name(0))

## 2. Extract the bundle

In [ ]:
import os, subprocess, pathlib

BUNDLE = '/content/drive/MyDrive/photoanalyzer_f1_bundle.tar'
WORK = pathlib.Path('/content/photoanalyzer')

assert os.path.exists(BUNDLE), (
    f'Bundle not found at {BUNDLE}. Upload it to the root of your Google Drive '
    'and re-run this cell.'
)
WORK.mkdir(exist_ok=True)

# Skip extraction if it's already there (speeds up re-runs after a disconnect).
if not (WORK / 'backend' / 'training_data').exists():
    print(f'Extracting {BUNDLE} to {WORK} …')
    subprocess.run(['tar', '-xf', BUNDLE, '-C', str(WORK)], check=True)
else:
    print('Bundle already extracted — skipping.')

n_images = sum(1 for _ in (WORK / 'backend' / 'training_data').rglob('*.jpg'))
n_ann = sum(1 for _ in (WORK / 'backend' / 'training_data_annotations').glob('*.json'))
print(f'{n_images} images, {n_ann} annotation JSONs available.')

## 3. Install Python dependencies

In [ ]:
!pip install -q -U 'transformers>=4.45' 'supervision>=0.24' 'torchvision>=0.20' 'Pillow>=10' 'tqdm>=4.66'

## 4. Restore prior outputs (resumable across disconnects)

If a prior Colab session got partway through, its `f1_outputs.tar` is in Drive. Pulling it back in means this session skips every image it already labelled.

In [ ]:
import os, subprocess

OUT_BUNDLE = '/content/drive/MyDrive/f1_outputs.tar'
BOXES_DIR = '/content/photoanalyzer/data/pseudo_labels/boxes'
os.makedirs(BOXES_DIR, exist_ok=True)

if os.path.exists(OUT_BUNDLE):
    print(f'Restoring previous outputs from {OUT_BUNDLE} …')
    subprocess.run(
        ['tar', '-xf', OUT_BUNDLE, '-C', '/content/photoanalyzer'],
        check=True,
    )
    n = len(os.listdir(BOXES_DIR))
    print(f'{n} previously-labelled images restored. Runner will skip them.')
else:
    print('No prior outputs — fresh run.')

## 5. Run the auto-labeler

`--shuffle` makes early progress span all sources (dakkadakka / reddit / cmon / ebay / isolation), so a mid-run check shows you diverse coverage instead of just whichever faction `rglob` hits first.

In [ ]:
%cd /content/photoanalyzer
!python scripts/phaseF/autolabel.py --shuffle

## 6. Save outputs to Drive

Writes `f1_outputs.tar` to the Drive root. If Colab disconnects mid-run, step 4 will restore from this bundle on the next session.

In [ ]:
import subprocess

print('Tarballing outputs …')
subprocess.run(
    [
        'tar', '-cf', '/content/drive/MyDrive/f1_outputs.tar',
        '-C', '/content/photoanalyzer',
        'data/pseudo_labels',
    ],
    check=True,
)
print('Saved /content/drive/MyDrive/f1_outputs.tar')
print('Download it from Drive to your repo root, then: tar -xf f1_outputs.tar')

## (Optional) 7. Periodic checkpoint while still running

Only useful if you're babysitting. Runs cell 6 on a timer so the Drive bundle stays fresh in case the runtime dies. Skip if you hit Run All and walked away — cell 6 still runs at the end.

In [ ]:
# Uncomment to enable periodic checkpointing alongside the runner.
# import threading, subprocess, time
# def checkpoint_loop(interval_min=30):
#     while True:
#         time.sleep(interval_min * 60)
#         try:
#             subprocess.run([
#                 'tar', '-cf', '/content/drive/MyDrive/f1_outputs.tar',
#                 '-C', '/content/photoanalyzer', 'data/pseudo_labels',
#             ], check=True)
#             print(f'[checkpoint {time.strftime("%H:%M")}] outputs mirrored to Drive')
#         except Exception as e:
#             print(f'[checkpoint err] {e}')
# threading.Thread(target=checkpoint_loop, daemon=True).start()